In [6]:
import numpy as np
import pandas as pd
import torch

# Encoding model
goal: input image -> features
The input dimension: B, C, H, W 
- B batch 
- C channel (for an RGB image = 3)
- H (height)
- W (width)

output: a representation (Z): B, D (embedding with dimension D)
- B batch
- D embedding/vector dimension (embedding is ViT lingo?)

We buidl our models using classes that inherit from torch.nn.Module

Think of nn.Module as a box/container that:
- Stores parameters (weights, biases)
- Knows how to apply itself to data
- Keeps track of submodules
- Enables automatic differentiation
- Plays nicely with GPUs, saving, loading, training modes

Each model has two absolutely necessary components:
- an init which defines the structure:
    - how many layers does the network have
    - what are the parameters of the model
    - and essentially what happens to the input in each layer
    - essentially, init builds the architecture but it doesn't run it

- forward which defines the computations done on the image
    - define how the data flows in the network, specifying the order of operations
    - essentially what happens when you hit "run"

# CNN
here we will build a simple convolutional neural network (CNN): a slide which would explain the CNN idea 

slides that would explain all the concepts like convolution 

- conv layers extract local patterns
- you also do a downsampling which builds invariance? _think like the visual stream going from v1 to v4_
- Global average pooling converts a feature map to a vector


In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# here we are building a very simple CNN model with 
# 3 convolutional layers
class SimpleCNN(nn.Module):
    """
    Image -> embedding encoder.
    Input:  (B, 3, H, W)  (assume RGB)
    Output: (B, D)
    """
    def __init__(self, emb_dim: int = 256):
        super().__init__() # inherrit from nn.module
        # define layers
        # we will use nn.Sequential to stack layers
        # Each layer consists of Conv2d + ReLU (NOTE: it can have other components like BatchNorm, Dropout, pooling, etc.)
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),  # (H/2, W/2)
            # output dimension after this layer is (B, 32, H/2, W/2)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # (H/4, W/4)
            # output dimension after this layer is (B, 64, H/4, W/4)
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),# (H/8, W/8)
            # output dimension after this layer is (B, 128, H/8, W/8)
            nn.ReLU(),
        )
        # finally a linear layer to get the embedding
        # when using nn.Linear, the output dimension comes second
        self.proj = nn.Linear(128, emb_dim)
        

    def forward(self, x):
        """forward pass through the network

        Args:
            x (_type_): input image tensor

        Returns:
            z (_type_): embedding vector for the input image
        """
        # x: (B, 3, H, W)
        f = self.conv(x)                 # (B, 128, h, w)
        f = f.mean(dim=(2, 3))           # global average pool -> (B, 128)
        z = self.proj(f)                 # (B, emb_dim)
        return z

# Vision transformers
What is a ViT conceptually?

turns an image into a sequence, processes the sequence with a transformer, outputs a representation. It's components are:

- patch embeddings
- positional encoding (this is not like CNNs that extract the local patterns, it first divides the image into small patches where each patch is processed separately so it needs to know where each patch is)
- transformer blocks
- pooling across patches (to get mean or cls tokens)
- projection head (???)

First we will build each component separately and then we will provide an all-in-one code/cell that gives you the model

## Patch embedding
It simply transforms the image into a sequence of patches (the location of the patches will be added as well later)

__STEP A__:Divides the image into smaller patches so if each image has the dimension HxW and your patches are PxP in size (PxPxC), you will have H/P and W/P patches across the image. NOTE that patches are non-overlapping

__STEP B__: convert each patch into an embedding (a fancy way of saying vector). So each (PxPxC) patch is **mapped** onto a vector with dimension D (embedding dimension). **This mapping is done with a learnable linear projection**. And because it is a learnable projection, patchEmbedding needs to inherit from torch.nn.Module (NOTE again that nn.Module is our way of telling pytorch that this is container that contains learnable parameters)

**IMPORTANT NOTE: patch embeddings are not just flattened pixel intensity values**
the mapping is by definition a linear affine transformation that contains no non-linearities

PatchEmbedding is not just about convenience (dim reduction) or shape compatibility. It actually tries to solve a problem which will make more sense when you consider a ViT as a whole. But for now:
PatchEmbedding solves this problem: How should local pixel information be expressed so that global reasoning (attention) becomes easy? **That is a representation problem, not a geometry problem.**


SO WHAT DOES THE PATCH EMBEDDING DO? (our contract)

- input: x with shape (B, C, H, W)
- output: tokens with shape (B, N, D) where:
    - N = number of patches
    - D = embedding dimension

- it contains learnable parameters. In other words it has weights (patch pixel -> embedding) that must be learnt


In [15]:
import torch
import torch.nn as nn

# why do we need PatchEmbedding to be a nn.Module? It contains leanable parameters.
# we are telling PyTorch to keep track of these parameters for us.
# and that this module contains learnable parameters and maybe even sub-modules.
# and we are telling PyTorch to keep track of these parameters for us.
# it is also a reusable component that can be easily integrated into larger models.
class PatchEmbedding(nn.Module):
    def __init__(
        self,
        img_size=224, # input image size (H and W are assumed to be equal)
        patch_size=16, # patch size (H and W will be divided by this number to get the number of patches)
        in_chans=3, # number of input channels (3 for RGB images)
        embed_dim=768, # embedding dimension (output dimension of each patch embedding
    ):
        super().__init__() # initialize the parent class nn.Module
        # a quick sanity check
        # NOTE that we are assuming non-overlapping patches here
        # Image must divide evenly into patches
        # why do we need assert here? 
        # to catch errors early and provide meaningful error messages.
        assert img_size % patch_size == 0, "Image size must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        # calculate number of patches
        # this is a metadata we might need later for cls tokens and positional embeddings
        # it is not a learnable parameter, so we just store it as a regular attribute
        self.num_patches = (img_size // patch_size) * (img_size // patch_size)

        # now the big part: defining the projection with learnable parameters
        # this is defined as a Conv2d layer 
        # to make sure that the patches are non-overlapping,
        # we set the kernel size and stride to be equal to patch_size
        self.proj = nn.Conv2d(
            in_channels=in_chans,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            bias=False, # no bias term? can be set to True if needed
        )
        # what parameters does this layer create?
        # self.proj.weight  # shape: (embed_dim, in_chans, patch_size, patch_size)
        # self.proj.bias    # None, since bias=False
    def forward(self, x):
        """forward pass through the Patch Embedding layer"""
        # as said earlier, this basically defines what happens when the input 
        # passes through the PatchEmbedding layer
        # apply the patch projection
        x = self.proj(x)  # shape: (B, embed_dim, H/patch_size, W/patch_size)
        # flatten the spatial dimensions, the spatial grid, (H/patch_size, W/patch_size) into one dimension
        x = x.flatten(2)  # shape: (B, embed_dim, num_patches)
        # transpose to get the final shape
        # the final shape would be (B, num_patches, embed_dim)
        x = x.transpose(1, 2)  # shape: (B, num_patches, embed_dim)
        return x


## Positional embedding
at this point, we have transformed/mapped the image from raw space to an embedding space. As we said this mapping is learnable! 

Now what are positional embedding? This section helps you understand what they are and why we need them. 

Let's consider an analogy (an imperfect analogy but helpful otherwise):

Imagine you take a photograph of a cat, cut it into 16 square patches, and throw those patches into a bag. If you hand that bag to a human, they can easily reconstruct the cat because they understand how ears, eyes, and paws relate **spatially**.

However, a Transformer—the architecture behind ViT—is "permutation invariant." Without help (which will come once you add the positional embedding), it treats those patches as a list of independent items. To a Transformer, the "ear patch" and the "tail patch" have no spatial relationship; it doesn't know which is top-left or bottom-right. It sees a "bag of patches," not an image.

To fix this, we must "tag" every patch with its location. But we don't just write a number on the back of the patch. Instead, we use Positional Embeddings: high-dimensional vectors that are added directly to the image data. But why can't we just use a number on the back of the patch. Consider this analogy:

The Radio Station Analogy: Think of the image data (color and texture) as a song being broadcast on a radio. If you want the listener to know which station they are tuned to (the position), you have two choices:

- The Low-D Way: You whisper the station ID over the music. It’s likely to be drowned out by a loud drum solo.

- The High-D Way (Positional Embeddings): You broadcast the station ID on a separate frequency at the same volume as the music. The receiver can "tune in" to the music and the ID simultaneously without them interfering.

By making the positional vector the same size as the image vector (e.g., 768 dimensions), we ensure the "**spatial signal**" is loud enough for the model to hear and use.

You might wonder: "Why not just use (x, y) coordinates like (0,1) or (0,2)?" Modern AI models use complex vectors (either learned through training or generated via sine/cosine functions) because they need to perform geometry via math.

Measuring Distance: Through a process called Self-Attention, patches "look" at each other. High-dimensional vectors allow the model to use dot products to mathematically calculate exactly how far apart two patches are.

Relative Relationships: These embeddings allow the model to understand that the relationship between an "eye" and a "nose" is the same, whether the cat is in the center of the frame or the corner.

You can get the positional embeddings in two ways:
- learnable positional embeddings
    - You treat the positions like any other weight in the neural network (as nn.Parameter)

    - How it works: You initialize a matrix of random numbers. During training, the model realizes, "Hey, every time I see a 'grass texture' at Index 140, it's usually at the bottom of the image." It then adjusts the values in the embedding for Index 140 to represent "bottom-ness."

    - Extremely flexible; the model can learn custom spatial biases (like focusing more on the center).

- Sinusoidal (Fixed) Positional Encoding
    - This uses fixed mathematical formulas to generate the vectors.
    - How it works: We use sine and cosine waves of different frequencies. Each dimension of the 768-D vector corresponds to a different wavelength.

    - The Logic: Just like a clock uses three hands (seconds, minutes, hours) to represent time, these waves allow the model to represent position. The "fast" waves represent local neighbors, and "slow" waves represent distant corners.

    - It doesn't need to be learned (saves memory). Theoretically, it can extrapolate to longer sequences (larger images) better than learnable ones.


The positional embeddings are "added" (not "concatenated") to the patch embeddings. But why?

If you have two different types of information (the What and the Where), it feels more intuitive to keep them separate by sticking them side-by-side (concatenation).However, adding them is not only more efficient but is actually mathematically "cleaner" for the way Transformers work. Here are the three main reasons why we add ($+$) instead of concatenate ($[\dots, \dots]$).

The model discovers its own way to separate the signals without you having to double the size of the vector. And we are working in a very high dimensional space (like 768 or 1024) and in a high dimensional space the model’s weight matrices are large enough to learn to "project" the sum back into two separate parts. It can learn to look at dimensions 1–384 for color/texture and dimensions 385–768 for position.

If you concatenate, you increase the size of the vector.

- Add: Patch (768) + Position (768) = 768 dimensions.
- Concatenate: Patch (768) + Position (768) = 1,536 dimensions.

If you double the dimensions, the number of parameters in your model doesn't just double—it quadruples in the linear layers (because a matrix is $dim \times dim$).

In [16]:
import torch
import torch.nn as nn

# Parameters
num_patches = 196  # e.g., 14x14 grid
embed_dim = 768    # Size of the "signal"

# 1. Define the 'Learnable' parameters
# We create a parameter of shape (1, num_patches, embed_dim)
# it is initialized randomly here, but it will be learned during training
pos_embed = nn.Parameter(torch.randn(1, num_patches, embed_dim))

# 2. To use it, you simply add it to your patch embeddings
# x = patch_embeddings + pos_embed

Putting together what we have so far:

In [17]:
import torch
import torch.nn as nn

# why do we need PatchEmbedding to be a nn.Module? It contains leanable parameters.
# we are telling PyTorch to keep track of these parameters for us.
# and that this module contains learnable parameters and maybe even sub-modules.
# and we are telling PyTorch to keep track of these parameters for us.
# it is also a reusable component that can be easily integrated into larger models.
class PatchEmbedding(nn.Module):
    def __init__(
        self,
        img_size=224, # input image size (H and W are assumed to be equal)
        patch_size=16, # patch size (H and W will be divided by this number to get the number of patches)
        in_chans=3, # number of input channels (3 for RGB images)
        embed_dim=768, # embedding dimension (output dimension of each patch embedding
    ):
        super().__init__() # initialize the parent class nn.Module
        # a quick sanity check
        # NOTE that we are assuming non-overlapping patches here
        # Image must divide evenly into patches
        # why do we need assert here? 
        # to catch errors early and provide meaningful error messages.
        assert img_size % patch_size == 0, "Image size must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        # calculate number of patches
        # this is a metadata we might need later for cls tokens and positional embeddings
        # it is not a learnable parameter, so we just store it as a regular attribute
        self.num_patches = (img_size // patch_size) * (img_size // patch_size)

        # now the big part: defining the projection with learnable parameters
        # this is defined as a Conv2d layer 
        # to make sure that the patches are non-overlapping,
        # we set the kernel size and stride to be equal to patch_size
        self.proj = nn.Conv2d(
            in_channels=in_chans,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            bias=False, # no bias term? can be set to True if needed
        )
        # what parameters does this layer create?
        # self.proj.weight  # shape: (embed_dim, in_chans, patch_size, patch_size)
        # self.proj.bias    # None, since bias=False

        # define the learnable positional embeddings
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, embed_dim))
    def forward(self, x):
        """forward pass through the Patch Embedding layer"""
        # as said earlier, this basically defines what happens when the input 
        # passes through the PatchEmbedding layer
        # apply the patch projection
        x = self.proj(x)  # shape: (B, embed_dim, H/patch_size, W/patch_size)

        # flatten the spatial dimensions, the spatial grid, (H/patch_size, W/patch_size) into one dimension
        x = x.flatten(2)  # shape: (B, embed_dim, num_patches)
        # transpose to get the final shape
        # the final shape would be (B, num_patches, embed_dim)
        x = x.transpose(1, 2)  # shape: (B, num_patches, embed_dim)

        # add positional embeddings
        x = x + self.pos_embed  # shape: (B, num_patches, embed_dim)
        return x


## CLS tokens
The [CLS] (short for "Classification") token is a special, "empty" vector that you prepend to your sequence of image patches before they enter the Transformer.


If you have 196 image patches, you actually feed the model 197 items. The first item (index 0) item is this special [CLS] token. NOTE that CLS token is preprended

The CLS token is a special, learnable token that is:

- **prepended** to the sequence of patch tokens (NOTE: PREPENDED)
- treated exactly like any other token by the Transformer
- later used as a global summary of the entire image

This analogy can help understand what it is: 
Imagine a corporate meeting where 196 department heads (the patches) are all talking to each other, sharing information about their specific area of the image.

The [CLS] token is like a secretary who enters the room with a blank notepad.

The secretary doesn't represent any specific department (they aren't a piece of the image).

Their only job is to listen and talk to everyone else. In other words, The CLS token both influences and is influenced by the other tokens.

By the end of the meeting (after passing through all Transformer layers), the secretary’s notepad contains a summary of the entire conversation.

When it’s time to decide if the image is a "Cat" or a "Dog," we don't ask the patches; we ask the secretary (the [CLS] token).

In a CNN, we usually perform "Global Average Pooling" at the end—mathematically averaging all the features to get one final result.

In a Transformer, we want something more sophisticated. Since the Transformer uses Self-Attention, every patch is constantly updating itself based on every other patch. By adding the [CLS] token:

- Neutral Gathering: It provides a dedicated "collection point" that isn't biased by being a "top-left corner" or "bottom-right corner."

- Global Context: Because of the Attention mechanism, the [CLS] token "attends" to every single patch. By the time it reaches the final layer, its vector has been influenced by every important feature found across the entire image.



Here's another way to think about why CLS tokens exist:
In CNNs the last step is global pooling. Unlike CNNs, Transformers do not have a fixed spatial hierarchy or an implicit “last layer” feature map. They do not have a built-in notion of “global pooling”.

There are two ways you can bake this notion of global pooling into the transformer:
The main question you need to answer here is: How do we turn a set of patch embeddings into one vector?

- option A: mean/max pooling: image → patches → transformer → average all tokens
- option B: using CLS tokens as learnable parameters: image → [CLS] + patches → transformer → take CLS




In [18]:
# 2️⃣ CLS token (learnable, shared across batch)
# Shape: (1, 1, embed_dim)
# cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

# this is defined as a learnable parameter inside the ViT model class
# this should be added before the positional embeddings


In [19]:
# Complete updated code: 
import torch
import torch.nn as nn

# why do we need PatchEmbedding to be a nn.Module? It contains leanable parameters.
# we are telling PyTorch to keep track of these parameters for us.
# and that this module contains learnable parameters and maybe even sub-modules.
# and we are telling PyTorch to keep track of these parameters for us.
# it is also a reusable component that can be easily integrated into larger models.
class PatchEmbedding(nn.Module):
    def __init__(
        self,
        img_size=224, # input image size (H and W are assumed to be equal)
        patch_size=16, # patch size (H and W will be divided by this number to get the number of patches)
        in_chans=3, # number of input channels (3 for RGB images)
        embed_dim=768, # embedding dimension (output dimension of each patch embedding
    ):
        super().__init__() # initialize the parent class nn.Module
        # a quick sanity check
        # NOTE that we are assuming non-overlapping patches here
        # Image must divide evenly into patches
        # why do we need assert here? 
        # to catch errors early and provide meaningful error messages.
        assert img_size % patch_size == 0, "Image size must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        # calculate number of patches
        # this is a metadata we might need later for cls tokens and positional embeddings
        # it is not a learnable parameter, so we just store it as a regular attribute
        self.num_patches = (img_size // patch_size) * (img_size // patch_size)

        # now the big part: defining the projection with learnable parameters
        # this is defined as a Conv2d layer 
        # to make sure that the patches are non-overlapping,
        # we set the kernel size and stride to be equal to patch_size
        self.proj = nn.Conv2d(
            in_channels=in_chans,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            bias=False, # no bias term? can be set to True if needed
        )
        # what parameters does this layer create?
        # self.proj.weight  # shape: (embed_dim, in_chans, patch_size, patch_size)
        # self.proj.bias    # None, since bias=False
        
        # add cls token
        # the dimensions are (1, 1, embed_dim)
        # meaning there is one shared cls token across the batch
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # define the learnable positional embeddings
        # num_patches + 1 to account for cls token
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches+1, embed_dim))
    def forward(self, x):
        """forward pass through the Patch Embedding layer"""
        # as said earlier, this basically defines what happens when the input 
        # passes through the PatchEmbedding layer
        # apply the patch projection
        x = self.proj(x)  # shape: (B, embed_dim, H/patch_size, W/patch_size)

        # flatten the spatial dimensions, the spatial grid, (H/patch_size, W/patch_size) into one dimension
        x = x.flatten(2)  # shape: (B, embed_dim, num_patches)
        # transpose to get the final shape
        # the final shape would be (B, num_patches, embed_dim)
        x = x.transpose(1, 2)  # shape: (B, num_patches, embed_dim)

        # Expand CLS token across batch
        # We want to prepend one CLS token to each image in the batch.
        # That means we need B copies of the CLS token one per image.
        # We now need to logically turn: (1, 1, embed_dim) -> (B, 1, embed_dim)
        B = x.shape[0]
        cls_tokens = self.cls_token.expand(B, -1, -1)
        # shape: (B, 1, embed_dim)

        # Prepend CLS token
        x = torch.cat((cls_tokens, x), dim=1)
        # shape: (B, num_patches + 1, embed_dim)

        # add positional embeddings
        x = x + self.pos_embed  # shape: (B, num_patches + 1, embed_dim)
        return x


## Transformer blocks


### Attention
the Attention Mechanism is the model's ability to "look around" that map and decide which parts of the landscape matter most for the task at hand.

Each token is a vector, but a token should not be processed in isolation.
A patch token should be able to “pull in” information from other patches.

Let's take a look at attention at a conceptual level first. 

The goal of Self-Attention is simple: for any given token (say, a patch containing a "cat's ear"), which other patches in the image provide the most useful context to understand it?

To understand an "ear," the model might attend heavily to the "eyes" or "fur" patches.

It will likely ignore the "sky" patches in the background.

ViTs do this mathematically using Query, key, and value. 
- **Query**: what am I looking for? 
- **Key**: what do I contain? what's my label?
- **Value**: what information do I have to offer?

Q, K, and V are linear projections of the X (input data) and so we implement them using nn.Linear. In other words:

Q = x @ W_Q → (B, N, D)

K = x @ W_K → (B, N, D)

V = x @ W_V → (B, N, D)

The Ws are estimated during training. 


Consider this analogy: You're at a library and you are looking for a specific information. You have your **Query**, you look around at the labels of the books (**keys**) and when you find a relevant book, you take the book to get the info you were looking book (the info in the book is **Value**)

Now let's go back to the transformers: 
- Each token/patch sends out its Query.
- It compares its Query against every other token's Key using a Dot Product.
- The result is an **Attention Score**. A high score means "These two are highly relevant to each other."
- The token then takes a weighted sum of the Values of the tokens it liked best.

Because this calculation happens for every pair of tokens, the model can connect a patch in the top-left corner with a patch in the bottom-right corner in a single step. This is why Transformers are so good at understanding complex scenes—they don't have to "wait" for information to travel through layers like a CNN does.

attention is calculated as: 

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$


Dimensions: 

Q ∈ ℝ^(B × N × D)

K ∈ ℝ^(B × N × D)

V ∈ ℝ^(B × N × D)

Now let's get an intuition about this formula:
- remember we said "_It compares its Query against every other token's Key using a Dot Product._" that part is done through ${QK^T}$, dot product of Query and key which can be considered as a similarity measure so you can think of it as comparing queries against keys. It’s a measure of "raw compatibility." If the vectors point in the same direction, the score is high.
    - this is attention score: tells you how relevant two patch tokens are
    - why $\sqrt{d_k}$? to handle two problems
        - When you calculate the Attention Score ($QK^T$), you are doing a dot product between two vectors of size $d_k$ (e.g., 768).Mathematically, if the elements of your $Q$ and $K$ vectors are random variables with a mean of 0 and a variance of 1, their dot product will have a mean of 0 but a variance of $d_k$.As $d_k$ grows (as our model gets "wider" and more powerful), the dot products can grow to very large positive or negative values.
        - The next step in the pipeline is the Softmax. The Softmax function is very sensitive to the magnitude of its input. If the input values are huge (e.g., 500 vs. -500), the Softmax "saturates." It pushes the highest value to 1.0 (100%) and everything else to 0.0.

- $\text{softmax}$ turns the values into probabilities summing to 1 (or %100)
    - $\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)$ this is the attention weight. 
    - attention weight: who is the most important token? with highest probability

- $V$: Multiplying by the content to get the final updated token.
    - this final stage $\text{Weights} \times V$ that gives us the attention output (updated patch)

### Self attention and multi head attention

What we have been talking about so far is self-attention. 

Self-attention is a learned, content-dependent way for elements in a set to exchange information with each other. It has "self" because Q, K, and V are all from the same set of patch tokens (same image) and there is no external memory or outside information. 

Imagine a room with N people (tokens).

Each person:
- listens to everyone else but not equally
- gives more weight to people who seem relevant

After listening, each person:
- updates their own understanding
- becomes a bit different than before

That’s self-attention.


Now **Multi-head attention**

What is a head?
A head is one self-attention operation with _its own way_ of comparing tokens.
Think of a head as:
- one “attention lens” through which tokens look at each other
- Each head can focus on different kinds of relationships.

It is important to note that an attention head is not a neuron, not a layer, nor a block. It is simply an attention operation/mechanism. So with each attention head, we can attend to one feature/information and that's why we need multiple attention heads. 

Why do we need multiple heads?

With multiple heads, 
Instead of one meeting where everyone talks about color,
you have:

- one group talking about shape
- one group talking about color
- one group talking about symmetry
- one group talking about spatial layout

Then you combine their conclusions.

***Multi-head attention lets the model attend to different types of relationships at the same time.***

Here's another analogy that might be helpful:

Instead of having one giant brain looking at the image, you split the work. If your embedding size is 768 and you have 12 heads, each head only looks at a "slice" of 64 dimensions ($768 / 12 = 64$).

The Goal: Different heads learn to look for different things.Head 1 might only care about edges.Head 2 might only care about color contrast.Head 3 might specifically look at the relationship between the [CLS] token and everything else.

Analogy: Imagine a crime scene. You don't just send one detective. You send a Ballistics Expert, a DNA Expert, and a Photographer. They all look at the same room (the image), but they are looking for very different clues.

We do NOT explicitly tell each head to care about a different thing.
Each head learns to care about different things because it has its own learned projections and is trained jointly on the task. In other words, differentiation must emerge implicitly.

If two heads did exactly the same thing:

they would be redundant

the model would gain nothing from having both

During training:

- redundancy is not useful
- gradients push heads to specialize

specialization improves performance

So the optimization pressure is:
“Don’t waste capacity by doing the same thing twice.”

***This is emergent specialization, not enforced specialization. Redundant heads do not reduce the loss efficiently, so gradients push heads toward complementary roles.***

Each head is initialized randomly, so right off the bat they are different and as training proceeds, their "useful differences" get amplified. 
NOTE that there is one loss and one backward pass so there is no head-specific optimization

Heads are initialized with different parameters, which means they start with slightly different similarity functions. But it is important to note that:

- initialization does not guarantee different views
- it only breaks symmetry
- training + optimization pressure does the real work



NOTE:
“Each head sees the full embedding, but compresses it into its own lower-dimensional representation.”

Each head asks:

“Which combinations of the original features matter for my notion of similarity?”

That’s why the projection is learned.


### MLP 
With self-attention each token talks to all the other tokens. Then it's time for the token to update itself using what it has learnt from all the other tokens. And that happens through the MLP 

MLP is a function that takes a vector and transforms it through a sequence of linear mappings and nonlinearities.

Usually there are two MLPs. In other words:

x → Linear → Nonlinearity → Linear → output

Every token (including the [CLS] token) passes through the same Multi-Layer Perceptron (MLP). This usually consists of two linear layers with a non-linear activation (like GELU) in between.

NOTE: MLP operates per token

While Attention is where tokens interact, the MLP is where each token processes the information it just gathered. It's "private study time" after a "group discussion."

An MLP does **feature transformation, not communication**.

- It does not look at other tokens
- It does not mix information across positions
- It operates on one vector at a time
- The MLP is applied independently to each token


### Residual connections (Skipp connections)
The idea hear is similar to ResNets (in a sense). 

## head, block, and layer

We talked about head, each head is an attention mechanism/operation. 
Now what is a block?

an attention block consists of:
- Multi-Head Attention (The detectives talking).

- MLP / Feed-Forward (The detectives thinking).

- LayerNorm & Residuals (The paperwork and filing system).


In a transformer, layer and block are often used to mean the same thing. 
A Vision Transformer (like ViT-Base) usually has 12 Layers. This means the data goes through 12 identical Blocks in a row.

The output of layer 1 becomes the input of layer 2 and so on... (vertical processing as opposed to horizontal which is moving through heads. NOTE: every single head sees ALL the tokens)

This is about evolution of features.

- Early Layers (Low-Level): The "detectives" in Layer 1 are looking at raw pixels and positional embeddings. They can only see simple things like "there is a red line here" or "there is a sharp edge there."

- Mid Layers (Mid-Level): The detectives in Layer 6 aren't looking at pixels anymore; they are looking at the results from the previous layers. They see that a "red line" and a "curve" from Layer 1 have come together to form a "circle."

- Late Layers (High-Level): By Layer 12, the detectives are looking at highly processed data. They see that the "circles" from Layer 6 have come together to form "the wheels of a car."


**In summary**

Heads allow the model to be wide (seeing many types of details at once).

Layers (Blocks) allow the model to be deep (building complex ideas out of simple ones).

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadSelfAttention(nn.Module):
    """
    Multi-head self-attention module.
    Input:  x of shape (B, N, D)
    Output:     shape (B, N, D)
    """

    def __init__(self, embed_dim: int, num_heads: int, attn_dropout: float = 0.1, proj_dropout: float = 0.1):
        """_summary_

        Args:
            embed_dim (int): embedding dimension
            num_heads (int): number of attention heads
            attn_dropout (float, optional): regularization! What percentage of attention weights to drop. Defaults to 0.1.
            proj_dropout (float, optional): regularization! What percentage of output projection weights to drop. Defaults to 0.1.
        """
        # we have multiple heads of self-attention
        # each head has its own set of learnable parameters
        # parameters are weight matrices for query, key, value, and output projection
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        # now we need to assert that embed_dim is divisible by num_heads
        assert embed_dim % num_heads == 0, "Embedding dimension must be divisible by number of heads."
        # why is it important?
        # because output of each head will have a dimension of embed_dim // num_heads
        # in other words, each attention head will take the whole embedding vector
        # and project it down to a smaller dimension of embed_dim // num_heads
        self.head_dim = embed_dim // num_heads

        # Attention uses dot products Q·K.
        # Dot products grow with dimension, so we scale by 1/sqrt(d) for stability.
        self.scale = 1.0 / math.sqrt(self.head_dim)

        # define the learnable parameters
        # we can use nn.Linear to define the weight matrices for query, key, value, and output projection
        # each of these layers will have its own weight matrix
        # One Linear layer to create Q, K, V together.
        # Input per token:  D
        # Output per token: 3D  (concatenated Q, K, V)
        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=False)

        # dropout for attention weights
        self.attn_dropout = nn.Dropout(attn_dropout)

        # After combining head outputs, we apply a final projection back to D.
        # WHY? because each head outputs a vector of size head_dim
        # and we have num_heads of them, so we need to combine them back to embed_dim
        # but why is 
        self.proj = nn.Linear(embed_dim, embed_dim, bias=True)

        # dropout for output projection
        self.proj_dropout = nn.Dropout(proj_dropout)

    def forward(self, x: torch.Tensor)-> torch.Tensor:

        """forward pass through the Multi-head self-attention module"""

        # x is expected to be (B, N, D)
        B, N, D = x.shape  # unpack batch size, number of tokens, embedding dimension

        # Compute Q, K, V for all tokens in one matrix multiplication.
        # qkv shape: (B, N, 3D) q, k, and v are concatenated along the last dimension
        qkv = self.qkv(x)

        # first Reshape to separate Q/K/V and heads:
        # (B, N, 3D) -> (B, N, 3, H, d)
        qkv = qkv.view(B, N, 3, self.num_heads, self.head_dim)

        # Permute dimensions so Q/K/V become the first index:
        # (B, N, 3, H, d) -> (3, B, H, N, d)
        qkv = qkv.permute(2, 0, 3, 1, 4)

        # Split into q, k, v:
        # each is shape (B, H, N, d)
        q, k, v = qkv[0], qkv[1], qkv[2]

        # first Compute attention scores:
        # For each head, scores = Q @ K^T
        # q: (B, H, N, d)
        # k.transpose(-2, -1): (B, H, d, N)
        # scores: (B, H, N, N)
        # note that self.scale is defined in __init__
        # it is 1/sqrt(head_dim)
        # why k.transfose(-2, -1)? 
        # because we want to do dot product between each query and all keys
        # for transpose, -2 refers to the second last dimension
        # -1 refers to the last dimension
        # .transpose simply swaps these two dimensions
        # a simple .T doesn't work here because tensors can have more than 2 dimensions
        # in this case, we have 4 dimensions: (B, H, N, d)
        # so with .transpose we need to specify which two dimensions to swap
        scores = (q @ k.transpose(-2, -1)) * self.scale

        # calculate attention weights by applying softmax to scores
        attn_weights = F.softmax(scores, dim=-1)

        # dropout on attention weights for regularization
        attn_weights = self.attn_dropout(attn_weights)

        # Weighted sum of values:
        # weights: (B, H, N, N)
        # v:       (B, H, N, d)
        # out:     (B, H, N, d)
        out = attn_weights @ v

        # Reorder so tokens come before heads:
        # (B, H, N, d) -> (B, N, H, d)
        # in pytorch, .transpose(dim0, dim1) swaps two dimensions
        # here we are swapping dimension 1 and 2
        out = out.transpose(1, 2)

        # Merge heads back into the full embedding dimension:
        # (B, N, H, d) -> (B, N, D)
        out = out.reshape(B, N, D)

        # Final linear projection to mix information across heads.
        out = self.proj(out)

        # dropout on output projection for regularization
        out = self.proj_dropout(out)

        return out
        

class MLP(nn.Module):
    """
    Transformer MLP / FeedForward network.
    Applied independently to each token (no token-to-token mixing).
    Think: with MHSA, we mix information across tokens;
    here, MLP is just a transformation applied to each token separately. 
    so no mixing happens here.
    Input:  (B, N, D)
    Output: (B, N, D)
    """
    # consider two linear layers with a non-linearity in between
    def __init__(self, embed_dim: int, mlp_ratio: float = 4.0, dropout: float =0.1):
        super().__init__()

        self.embed_dim = embed_dim
        # Hidden dimension is typically 4x embedding dim in Transformers (a common default).
        hidden_dim = int(embed_dim * mlp_ratio)

        # note that the MLP is simply a bunch of linear layers with non-linearities in between
        # First linear expands the feature dimension: D -> hidden_dim.
        self.fc1 = nn.Linear(embed_dim, hidden_dim)

        # GELU is a smooth nonlinearity commonly used in Transformers.
        # this can also be ReLU, LeakyReLU, etc.
        self.act = nn.GELU()

        # now the regularization after the activation
        self.dropout1 = nn.Dropout(dropout)

        # now the second layer to project back to embed_dim
        self.fc2 = nn.Linear(hidden_dim, embed_dim)

        # why is there no activation after the second linear layer?
        # The second linear layer in the MLP is meant to be a projection, 
        # not a feature extractor, so we do not put a nonlinearity after it.
        # so its main job is to map the hidden representation back to the original embedding dimension.
        # expand → nonlinearity → compress

        # another dropout after the second linear layer
        self.dropout2 = nn.Dropout(dropout)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """forward pass through the MLP module"""
        # x is expected to be (B, N, D)
        x = self.fc1(x)          # (B, N, hidden_dim)
        x = self.act(x)         # (B, N, hidden_dim)
        x = self.dropout1(x)    # (B, N, hidden_dim)
        x = self.fc2(x)         # (B, N, D)
        x = self.dropout2(x)    # (B, N, D)
        return x


class TransformerEncoderBlock(nn.Module):
    """
    A full Transformer encoder block (ViT-style), using Pre-LayerNorm:
      x = x + Attention(LN(x))
      x = x + MLP(LN(x))

    Input:  (B, N, D)
    Output: (B, N, D)
    """
    # this is the full block with MSA and MLP with skip connections and layer norms
    def __init__(self,
        embed_dim: int,
        num_heads: int,
        mlp_ratio: float = 4.0,
        attn_drop: float = 0.0,
        proj_drop: float = 0.0,
        mlp_drop: float = 0.0,
    ):
        super().__init__()

        # LayerNorm normalizes each token's features (across D).
        # Pre-LN means we normalize BEFORE attention/MLP.
        # this is a layer normalization layer (not a batch normalization)
        # meaning this will do a normalization across the feature dimension for each token
        # as opposed to batch normalization which normalizes across the batch dimension
        self.norm1 = nn.LayerNorm(embed_dim)

        # now create an instance of MultiHeadSelfAttention
        # Multi-head self-attention sublayer.
        self.attn = MultiHeadSelfAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            attn_drop=attn_drop,
            proj_drop=proj_drop,
        )

        # note that you do not hit run here yet. You are just providing the recepie

        # Second LayerNorm before the MLP sublayer.
        self.norm2 = nn.LayerNorm(embed_dim)

        # now create an instance of MLP
        # MLP / FeedForward sublayer.
        self.mlp = MLP(embed_dim=embed_dim, mlp_ratio=mlp_ratio, drop=mlp_drop)

        # NOTE: skip connections/residual connections are implemented in the forward pass
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """forward pass through the Transformer Encoder Block"""
        # x is expected to be (B, N, D) 
        # ---- Attention sublayer (Pre-LN + residual) ----
        x = x + self.attn(self.norm1(x))
        # Explanation:
        # 1) self.norm1(x): normalize token features (stabilizes learning)
        # 2) self.attn(...): tokens mix information via attention
        # 3) x + ...       : residual connection keeps original info and adds an update

        # ---- MLP sublayer (Pre-LN + residual) ----
        x = x + self.mlp(self.norm2(x))
        # Explanation:
        # 1) self.norm2(x): normalize again before MLP
        # 2) self.mlp(...): per-token nonlinear feature transformation
        # 3) x + ...      : residual connection again
        return x

# BUILD A FULL VISION ENCODER

A full vision transformer encoder is comprised of all the modules and multiple heads

This is the flow of information:

IMAGE (B, 3, H, W)

   ↓

PatchEmbedding

   ↓

TOKENS (B, N, D)

   ↓

TransformerEncoderBlock × L

   ↓

TOKENS (B, N, D)

What do we have so far??

You already built:

✅ PatchEmbedding

input: images (B, 3, H, W)

output: tokens (B, N, D)

✅ MultiHeadSelfAttention

input: tokens (B, N, D)

output: tokens (B, N, D)

✅ MLP

input: tokens (B, N, D)

output: tokens (B, N, D)

✅ TransformerEncoderBlock

input: tokens (B, N, D)

output: tokens (B, N, D)

Now the missing step:

We never connected these pieces into a single encoder. An encoder is a **container** that wires layers together.

So our job now is to:

create a new nn.Module whose only job is to connect existing modules in the right order

In [ ]:
# build a full transformer encoder with multiple blocks
class VisionEncoder(nn.Module):
    """
    Full ViT-style encoder:
      image -> patch embedding -> transformer blocks -> tokens

    This module:
    1) takes an image
    2) converts it to patch tokens
    3) passes tokens through Transformer blocks
    4) returns token representations

    - This is not attention
    - This is not an MLP
    **This is architecture wiring (you build the components first and then wire them together)**
    """
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_chans=3,
        embed_dim=384,
        num_heads=12,
        mlp_ratio=4.0,
        depth=1, # how many Transformer encoder blocks you stack sequentially.
        attn_drop=0.1,
        proj_drop=0.1,
        mlp_drop=0.1,
    ):
        super().__init__()

        # 1) Patch embedding: images -> tokens
        self.patch_embed = PatchEmbedding(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
        )

        # 2) Stack of Transformer blocks
        # depth specifies how many blocks to stack
        # ModuleList is used to hold a list of sub-modules
        # telling pyTorch to keep track of these sub-modules and their parameters
        # here we only declare that these blocks exist
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                attn_drop=attn_drop,
                proj_drop=proj_drop,
                mlp_drop=mlp_drop,
            )
            for _ in range(depth)
        ])

    def forward(self, x):
        """
        x: (B, 3, H, W)
        returns: (B, N, D)
        """
        # images -> tokens
        x = self.patch_embed(x)      # (B, N, D)

        # tokens -> tokens
        for block in self.blocks:
            x = block(x)

        return x
